In [1]:
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = "1"

import torch
import matplotlib.pyplot as plt

from utils import *
from plot import *


/home/karanjot/.conda/envs/sae/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


config

In [2]:
base_f = './instruct'
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_id = 'meta-llama/Llama-3.1-8B-Instruct'
sae_id = './llama_scope/Llama3_1-8B-Base-L6R-8x'


In [3]:
generate = generate(model_id, sae_id, base_f, device)

`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 4/4 [00:00<00:00, 112.20it/s]


In [ ]:
tasks = ['privacy_protection', 'personalization', 'prioritization']

generate.load_data(data_f='instruct.json', tasks=tasks)


: 

In [4]:
generate.subset[6 : 8]

AttributeError: 'generate' object has no attribute 'subset'

In [ ]:
# pred = generate.run(out_f='prediction.json')

predictive inference: 100%|██████████| 311/311 [09:12<00:00,  1.78s/it]

results saved: ./instruct/prediction.json


In [5]:
# generate.ids

ids = generate.load_results(data_f='prediction.json')

In [6]:
task = 'privacy_protection'

for i in ['pass', 'fail']:
    print(f"{i}: {len(ids[task][i])}")


pass: 53
fail: 58


In [8]:
# generate.load_data(data_f='instruct.json', tasks=tasks)

# _ = generate.run_sae(layer=6, out_f='activations_topk',)

In [7]:
instances = {}

In [8]:
for i in ['pass', 'fail']:
    processed_instances = []

    for conv in ids[task][i]:
        conv = torch.load(os.path.join(base_f, f'activations_topk/{conv}.pt'))
        out = {'id': conv['id'], 'turns': []}

        for t_data in conv['results']:
            rep = get_turn_representation(
                t_data=t_data,
                source='user',
                selector='all',
                pooling='mean',
                lexical_only=True,
                spl_tokens=None,
            )

            out['turns'].append({
                'turn': t_data['turn'],
                'representation': rep,
            })
        processed_instances.append(out)

    instances[i] = processed_instances


In [9]:
# instances

In [10]:
instances['pass'][0]

{'id': '20f4f58f_2',
 'turns': [{'turn': 1,
   'representation': tensor([ 0.0060,  0.0175, -0.0296,  ..., -0.0177,  0.0050, -0.0164])},
  {'turn': 2,
   'representation': tensor([ 0.0578, -0.0079, -0.0942,  ..., -0.0452,  0.0138, -0.0209])}]}

In [11]:
instances_stats = {}

for i in ['pass', 'fail']:
    instances_stats[i] = get_stats(instances[i], level='turn')


In [12]:
chart = plot_concepts(instances_stats, groups=[1, 2], order_by=1, top_k=50, stat='median')

chart.save('turn_mean.html')